<a href="https://colab.research.google.com/github/crezny/DATASCI266_final_project/blob/main/LoRA Train.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%pip install -q -U torchinfo
%pip install -q -U bitsandbytes
%pip install -q -U accelerate
%pip install -U bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.4/59.4 MB 41.0 MB/s eta 0:00:00


In [2]:
import pandas as pd
from datetime import datetime, time
import re

import math, os

from typing import List, Dict, Any
import pandas as pd
import textwrap
import json
import time
import torchgen
import torch
from typing import Optional, Iterable, Dict, Any, List
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
    Trainer,
    BitsAndBytesConfig
)
from peft import get_peft_config, PeftModel, PeftConfig, get_peft_model, LoraConfig, TaskType
from datasets import (
    Dataset,
    load_dataset
)
from torchinfo import summary
from datetime import datetime, time
from typing import List, Dict, Any
from dotenv import load_dotenv
from sqlalchemy import create_engine, text
from sqlalchemy.exc import SQLAlchemyError
from typing import Optional, Iterable, Dict, Any, List


try:
    from tqdm.notebook import tqdm
except Exception:
    from tqdm import tqdm

try:
    from google.colab import userdata
    get_secret = userdata.get
except ImportError:
    # Fallback: define a dummy get_secret for local use
    def get_secret(key):
        return None

from dotenv import load_dotenv
load_dotenv()


from sqlalchemy import create_engine, text
from sqlalchemy.exc import SQLAlchemyError

def get_env(key: str, default: str = None):
    """Tries os.getenv first, then Colab userdata if available."""
    return os.getenv(key) or get_secret(key) or default

DB_HOST = get_env("POSTGRES_DB_HOST")
DB_PORT = int(get_env("POSTGRES_DB_PORT"))
DB_NAME = get_env("POSTGRES_DB_NAME")
DB_USER = get_env("POSTGRES_DB_USER")
DB_PASS = get_env("POSTGRES_DB_PASS")

connection_url = f"postgresql+psycopg2://{DB_USER}:{DB_PASS}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine = create_engine(connection_url, echo=False)



In [3]:
train_df = pd.read_sql("""
SELECT prompt_responses.*, policy_splits.split_label, policy.org_text FROM prompt_responses
JOIN policy_splits on policy_splits.policy_id = prompt_responses.policy_id
JOIn(
SELECT DISTINCT policy_id, section_id, section_title, source_framework, string_agg(clause_text, '\n' ORDER BY ROW_NUMBER) AS org_text  FROM policy_lines
GROUP BY policy_id, section_id, section_title, source_framework
) as policy on prompt_responses.section_id = policy.section_id
WHERE version = 'v2.5' AND split_label = 'train'""", engine)

test_df = pd.read_sql("""
SELECT prompt_responses.*, policy_splits.split_label, policy.org_text FROM prompt_responses
JOIN policy_splits on policy_splits.policy_id = prompt_responses.policy_id
JOIn(
SELECT DISTINCT policy_id, section_id, section_title, source_framework, string_agg(clause_text, '\n' ORDER BY ROW_NUMBER) AS org_text  FROM policy_lines
GROUP BY policy_id, section_id, section_title, source_framework
) as policy on prompt_responses.section_id = policy.section_id
WHERE version = 'v2.5' AND split_label = 'test'""", engine)

val_df = pd.read_sql("""
SELECT prompt_responses.*, policy_splits.split_label, policy.org_text FROM prompt_responses
JOIN policy_splits on policy_splits.policy_id = prompt_responses.policy_id
JOIn(
SELECT DISTINCT policy_id, section_id, section_title, source_framework, string_agg(clause_text, '\n' ORDER BY ROW_NUMBER) AS org_text  FROM policy_lines
GROUP BY policy_id, section_id, section_title, source_framework
) as policy on prompt_responses.section_id = policy.section_id
WHERE version = 'v2.5' AND split_label = 'val'""", engine)

In [4]:
model_name = "google/gemma-2-9b-it"
tokenizer = AutoTokenizer.from_pretrained(model_name)


tokenizer_config.json:   0%|          | 0.00/47.0k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/4.24M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

In [28]:
len(hard_cases_final)

200

In [29]:
import pandas as pd

sql = """
SELECT *
FROM sme_quality_score
WHERE split_label = 'train';
"""

filter_df = pd.read_sql(sql, engine)
high_quality = filter_df[filter_df["sme_structure_score"] >= 0.70].copy()
hard_cases   = filter_df[(filter_df["sme_structure_score"] >= 0.50) &
                  (filter_df["sme_structure_score"] < 0.70)].copy()
exclude      = filter_df[filter_df["sme_structure_score"] < 0.50].copy()

print("High-quality:", len(high_quality))
print("Hard cases:  ", len(hard_cases))
print("Exclude:     ", len(exclude))


import numpy as np

hard_cases = hard_cases.copy()
hard_cases["direct_frac"] = np.where(
    hard_cases["atomic_total"] > 0,
    hard_cases["atomic_direct_count"] / hard_cases["atomic_total"],
    0.0,
)

# Only keep rows that actually have mappings
hc = hard_cases[hard_cases["atomic_total"] > 0].copy()

# Compute data-driven thresholds
q_direct = hc["direct_frac"].quantile(0.25)          # bottom 25% direct_frac dropped
q_sim    = hc["atomic_sim_mean"].quantile(0.25)      # bottom 25% similarity dropped
q_req    = hc["sme_req_sent_ratio"].quantile(0.25)   # bottom 25% req density dropped

hc = hc[
    (hc["direct_frac"]       >= q_direct) &
    (hc["atomic_sim_mean"]   >= q_sim)    &
    (hc["sme_req_sent_ratio"]>= q_req)
].copy()


# Min-max normalize within hc to [0, 1]
def norm(col):
    c = hc[col]
    return (c - c.min()) / (c.max() - c.min() + 1e-6)

hc["norm_struct"] = norm("sme_structure_score")
hc["norm_direct"] = norm("direct_frac")
hc["norm_sim"]    = norm("atomic_sim_mean")

# Composite quality score inside hard_cases
hc["hard_quality_score"] = (
    0.5 * hc["norm_struct"] +
    0.3 * hc["norm_direct"] +
    0.2 * hc["norm_sim"]
)

# Pick the top N hard cases (tune N as you like)
N_HARD = 200  # or 150, 250, etc.
hard_cases_final = (
    hc.sort_values("hard_quality_score", ascending=False)
      .head(N_HARD)
      .copy()
)


lora_train_df = pd.concat(
    [high_quality, hard_cases_final],
    ignore_index=True
)

print("High-quality:", len(high_quality))
print("Hard cases (selected):", len(hard_cases_final))
print("Final LoRA train size:", len(lora_train_df))


High-quality: 50
Hard cases:   467
Exclude:      521
High-quality: 50
Hard cases (selected): 200
Final LoRA train size: 250


In [34]:
train_df = train_df[train_df['section_id'].isin(lora_train_df['section_id'].tolist())]

In [35]:
def build_chat_text(prompt, answer):
    messages = [
        {"role": "user", "content": prompt},
        {"role": "assistant", "content": answer},
    ]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,  # we want full input+target in training
    )
    return text

train_df["text"] = train_df.apply(
    lambda r: build_chat_text(r["system_prompt"], r["org_text"]),
    axis=1,
)
val_df["text"] = val_df.apply(
    lambda r: build_chat_text(r["system_prompt"], r["org_text"]),
    axis=1,
)

In [36]:
from datasets import Dataset

train_ds = Dataset.from_pandas(train_df[["text"]])
val_ds   = Dataset.from_pandas(val_df[["text"]])

max_length = 4096  # adjust if needed

def tokenize_fn(batch):
    out = tokenizer(
        batch["text"],
        max_length=max_length,
        truncation=True,
        padding="max_length",
    )
    # Standard causal LM: labels = input_ids
    out["labels"] = out["input_ids"].copy()
    return out

train_tok = train_ds.map(tokenize_fn, batched=True, remove_columns=["text"])
val_tok   = val_ds.map(tokenize_fn,   batched=True, remove_columns=["text"])


Map:   0%|          | 0/250 [00:00<?, ? examples/s]

Map:   0%|          | 0/117 [00:00<?, ? examples/s]

In [37]:
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
)

base_model = prepare_model_for_kbit_training(base_model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
)

model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()



config.json:   0%|          | 0.00/857 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/39.1k [00:00<?, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

model-00003-of-00004.safetensors:   0%|          | 0.00/4.96G [00:00<?, ?B/s]

model-00004-of-00004.safetensors:   0%|          | 0.00/3.67G [00:00<?, ?B/s]

model-00001-of-00004.safetensors:   0%|          | 0.00/4.90G [00:00<?, ?B/s]

model-00002-of-00004.safetensors:   0%|          | 0.00/4.95G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/173 [00:00<?, ?B/s]

trainable params: 54,018,048 || all params: 9,295,724,032 || trainable%: 0.5811


In [38]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [39]:
output_dir  = "/content/drive/MyDrive/266/gemma2_9b_it_v2_5_lora"


In [40]:
from transformers import TrainingArguments, Trainer
from transformers import DataCollatorForLanguageModeling

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,
)

training_args = TrainingArguments(
    output_dir="./gemma2_9b_it_v2_5_lora",
    per_device_train_batch_size=1,   # bump if VRAM allows
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,   # effective batch size = 8
    learning_rate=2e-4,
    num_train_epochs=3,
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",
    logging_steps=20,
    eval_strategy="steps",
    eval_steps=200,
    save_strategy="steps",
    save_steps=200,
    save_total_limit=3,
    bf16=torch.cuda.is_available(),
    gradient_checkpointing=True,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=val_tok,
    data_collator=data_collator,
)

trainer.train()
trainer.save_model(output_dir)
tokenizer.save_pretrained(output_dir)

`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


Step,Training Loss,Validation Loss


('/content/drive/MyDrive/266/gemma2_9b_it_v2_5_lora/tokenizer_config.json',
 '/content/drive/MyDrive/266/gemma2_9b_it_v2_5_lora/special_tokens_map.json',
 '/content/drive/MyDrive/266/gemma2_9b_it_v2_5_lora/chat_template.jinja',
 '/content/drive/MyDrive/266/gemma2_9b_it_v2_5_lora/tokenizer.model',
 '/content/drive/MyDrive/266/gemma2_9b_it_v2_5_lora/added_tokens.json',
 '/content/drive/MyDrive/266/gemma2_9b_it_v2_5_lora/tokenizer.json')

In [41]:
!cd "/content/drive/MyDrive/266" && zip -r gemma_lora_final.zip gemma2_9b_it_v2_5_lora


  adding: gemma2_9b_it_v2_5_lora/ (stored 0%)
  adding: gemma2_9b_it_v2_5_lora/README.md (deflated 65%)
  adding: gemma2_9b_it_v2_5_lora/adapter_model.safetensors (deflated 7%)
  adding: gemma2_9b_it_v2_5_lora/adapter_config.json (deflated 59%)
  adding: gemma2_9b_it_v2_5_lora/chat_template.jinja (deflated 52%)
  adding: gemma2_9b_it_v2_5_lora/tokenizer_config.json (deflated 96%)
  adding: gemma2_9b_it_v2_5_lora/special_tokens_map.json (deflated 76%)
  adding: gemma2_9b_it_v2_5_lora/tokenizer.model (deflated 51%)
  adding: gemma2_9b_it_v2_5_lora/tokenizer.json (deflated 84%)
  adding: gemma2_9b_it_v2_5_lora/training_args.bin (deflated 53%)
